# Introduction to Embeddings

Machine learning models cannot read strings directly — they operate on **numbers**. For language, the central question is: how do we turn text into numerical representations that preserve **meaning**?

**Embeddings** are dense vectors: fixed-length arrays of floats where semantically similar text ends up **geometrically close**. That geometry powers semantic search ("find docs like this question"), clustering, recommendations, deduplication, and the retrieval step in **retrieval-augmented generation (RAG)**.

Classical NLP introduced **word embeddings** (Word2Vec, GloVe) — one vector per word type. Modern GenAI systems use **API embedding models** that map entire sentences or document chunks to vectors in a single call, trained on web-scale data and optimized for downstream similarity.

This notebook builds intuition from sparse text representations, explains how dense embeddings differ, traces the evolution to today's API models, walks through the embedding lifecycle in applications, and runs live OpenAI embedding demonstrations.

Prerequisites: **02. GenAI & LLM Fundamentals**, **03. LLM API Integration**, and **AI Foundations → Bag of Words / TF-IDF** and **Word Embeddings** (for historical context).

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)

---

## 1. The Problem with Sparse Text Representations

Before embeddings went mainstream, text was often encoded as **sparse vectors** — high-dimensional, mostly zeros.

### 1.1 One-Hot and Bag-of-Words

**One-hot** encoding assigns each word in a vocabulary its own axis. Only one coordinate is 1; the rest are 0. **Bag-of-words (BoW)** counts how many times each word appears in a document. Both discard word order (BoW) and treat every word as unrelated to every other word (orthogonality).

| Issue | Consequence |
|-------|-------------|
| **High dimensionality** | Vocabulary $V$ can exceed 100,000 — vectors are huge |
| **Sparsity** | Most entries are zero; little information per dimension |
| **No similarity** | $\text{sim}(\text{cat}, \text{dog}) = 0$ in one-hot space |
| **No paraphrase** | "password reset" and "forgot login" share no features |
| **Out-of-vocabulary** | New words get zero vectors or are dropped |

Embeddings compress language into **dense** vectors (typically hundreds to a few thousand dimensions) where **similar meanings sit near each other**.

In [ ]:
vocab = ["cat", "dog", "car", "economics"]
word2idx = {w: i for i, w in enumerate(vocab)}


def one_hot(word: str, size: int = len(vocab)) -> np.ndarray:
    vec = np.zeros(size)
    vec[word2idx[word]] = 1.0
    return vec


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print("One-hot vectors (orthogonal axes):")
for w in vocab:
    print(f"  {w:12s} {one_hot(w).astype(int)}")

print(f"\ncosine(cat, dog)  = {cosine(one_hot('cat'), one_hot('dog')):.1f}")
print(f"cosine(cat, economics) = {cosine(one_hot('cat'), one_hot('economics')):.1f}")

In one-hot space, **cat** and **dog** are as unrelated as **cat** and **economics** — both cosine similarities are 0. Real language is not like that.

---

## 2. What Is an Embedding?

An **embedding** maps an object (word, sentence, image, user) to a vector $\mathbf{e} \in \mathbb{R}^d$ where $d$ is the **embedding dimension**.

$$\text{text} \;\mapsto\; \mathbf{e} = [e_1, e_2, \ldots, e_d]$$

For OpenAI `text-embedding-3-small`, $d = 1536$ by default. Every input — whether one word or a paragraph — returns a vector of that length.

### 2.1 Properties of Useful Text Embeddings

| Property | Meaning |
|----------|---------|
| **Semantic similarity** | Paraphrases have high cosine similarity |
| **Topic structure** | Related domains cluster in vector space |
| **Fixed dimensionality** | Same length for all inputs to a given model |
| **Opacity** | Individual dimensions are not human-readable labels |

You do not interpret $e_{42} = 0.09$ as "security feature on." You compare **whole vectors** using distance or similarity metrics (notebook 02).

### 2.2 The Distributional Idea (Brief)

> *"You shall know a word by the company it keeps."* — J.R. Firth, 1957

The **distributional hypothesis**: words (and phrases) that appear in similar contexts tend to have similar meanings. Embedding algorithms — from Word2Vec to transformer encoders — exploit co-occurrence and prediction objectives at scale.

See **AI Foundations → Word Embeddings** for Word2Vec, GloVe, and vector arithmetic (`king - man + woman ≈ queen`). This section focuses on **sentence-level API embeddings** used in production GenAI.

---

## 3. Evolution: From Word Vectors to API Embeddings

| Era | Representation | Granularity | Typical use |
|-----|----------------|-------------|-------------|
| **Sparse** | BoW, TF-IDF | Document | Classical ML classifiers |
| **Static word** | Word2Vec, GloVe | Word | Similar words, analogies |
| **Contextual** | BERT-style encoders | Token (context-dependent) | NLU benchmarks |
| **API embeddings** | `text-embedding-3-*` | Sentence / chunk | Search, RAG, clustering |

### 3.1 Static vs Contextual

| Type | Behavior | Example issue |
|------|----------|---------------|
| **Static** | One vector per word type | "bank" (river) vs "bank" (finance) share one vector |
| **Contextual** | Vector depends on surrounding text | Same word, different sentence → different representation |

API embedding models read your **full input string** and return **one vector for that span**. The word "bank" in different sentences produces different vectors because the model encodes context.

### 3.2 Word vs Sentence vs Document Embeddings

| Level | What you embed | RAG note |
|-------|----------------|----------|
| **Word** | Single token | Rarely used alone in RAG |
| **Sentence / chunk** | Paragraph section | **Default** — retrieve chunks |
| **Whole document** | Entire PDF | Often too coarse; split first (notebook 04) |

**Rule of thumb for RAG:** embed at the same granularity you **retrieve**.

---

## 4. Keyword Search vs Semantic Search

### 4.1 Keyword (Lexical) Search

Matches literal tokens — BM25, SQL `LIKE`, inverted indexes. Fast, exact for SKUs and error codes. Fails on paraphrase and synonyms.

### 4.2 Semantic (Vector) Search

Embed query and documents; rank by vector similarity. Handles paraphrase. Can miss exact rare tokens unless combined with keywords (**hybrid search**, notebook 08).

| Query | Document snippet | Keyword | Semantic |
|-------|------------------|---------|----------|
| "password reset" | "forgot my login credentials" | Weak | Strong |
| "error ECONNREFUSED" | "error ECONNREFUSED" | Strong | Strong |
| "refund policy" | "money-back guarantee" | Partial | Strong |

Production systems often use **both** — vectors for meaning, keywords for precision.

In [ ]:
# Toy keyword scorer (no API) — same idea as 08. RAG Fundamentals/01_naive_rag.py
DOCS = [
    "Use the password reset link on the login page.",
    "JWT tokens expire after 15 minutes.",
    "Alembic runs database schema migrations.",
]

query = "forgot my login"
q_words = set(query.lower().split())

print("Keyword overlap scores:")
for doc in DOCS:
    overlap = len(q_words & set(doc.lower().split()))
    print(f"  {overlap}  {doc}")

The password-reset doc may score **0** on keyword overlap for "forgot my login" — semantic embeddings fix that class of miss.

---

## 5. The Embedding Lifecycle in Applications

In a RAG or search product, embeddings appear in a repeatable pipeline:

```
  INGEST (offline / batch)          QUERY (online / per request)
  ───────────────────────          ────────────────────────────
  Raw documents                    User question
       ↓                                ↓
  Chunking                         Embed query
       ↓                                ↓
  Embed each chunk                 Vector similarity search
       ↓                                ↓
  Store in vector DB               Top-k chunks + metadata
                                         ↓
                                   LLM prompt with context (RAG)
```

| Stage | Notebook in this section |
|-------|--------------------------|
| Chunking | 04. Document Chunking |
| Embedding API | 03. Embedding APIs |
| Similarity math | 02. Similarity Search |
| Vector storage | 05–07. Vector DBs & Chroma |
| Retrieval eval | 09. Evaluating Retrieval |
| LLM answer | **08. RAG Fundamentals** |

This section covers everything **up to** the LLM generation box.

---

## 6. How API Embedding Models Work (High Level)

You do not train these models in application code. You call an API:

```python
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=["text to embed"],
)
vector = response.data[0].embedding
```

Internally, a **transformer encoder** reads tokens, builds contextual representations, and **pools** them (e.g. mean pooling or a dedicated embedding head) into one output vector per input string.

### 6.1 OpenAI Embedding Models (Starting Points)

| Model | Typical $d$ | Use |
|-------|-------------|-----|
| `text-embedding-3-small` | 1536 (configurable lower) | Default — learning & high-volume RAG |
| `text-embedding-3-large` | 3072 | Higher quality retrieval |
| `text-embedding-ada-002` | 1536 | Legacy systems |

Billing is per **input token** embedded. Long texts must be **chunked** before embedding (notebook 04).

### 6.2 What You Control vs What You Do Not

| You control | Provider controls |
|-------------|-------------------|
| Input text, batching | Model weights |
| Model name, `dimensions` | Training data & objectives |
| When to re-embed | Model version updates |

---

## 7. Common Use Cases

### 7.1 Semantic Search

Enterprise knowledge bases, support portals, and code search. Users ask in natural language; the system returns relevant passages without exact keyword overlap.

### 7.2 RAG (Retrieval-Augmented Generation)

Retrieve top-k chunks → insert into LLM prompt → model answers with grounded context. Embeddings are the **retrieval** mechanism; prompting is the **generation** mechanism (**04. Prompt Engineering**).

### 7.3 Clustering and Exploration

Embed thousands of support tickets; cluster to discover recurring themes. Cheaper than reading every ticket manually.

### 7.4 Deduplication and Near-Duplicate Detection

If $\text{cosine}(\mathbf{e}_1, \mathbf{e}_2) > 0.95$, flag as near-duplicate FAQ entries or redundant docs.

### 7.5 Recommendations

"Users who read this article also read..." — embed articles or products; recommend nearest neighbors to the current item.

---

## 8. Limitations and Misconceptions

| Misconception | Reality |
|---------------|---------|
| "Embeddings understand truth" | They encode **statistical similarity**, not factual verification |
| "One vector per PDF is enough" | Large docs need **chunking** or retrieval is vague |
| "Similarity = relevance" | High cosine can be same topic but wrong answer |
| "Embeddings are private" | Store and treat like source text; not encryption |
| "Set and forget" | Model or chunk changes require **re-indexing** |

Always evaluate retrieval on labeled queries (notebook 09) before blaming the LLM for bad RAG answers.

---

## 9. Mental Model for Debugging

When search or RAG fails, ask:

1. **Chunking** — Is the answer text split across chunks or buried in noise?
2. **Embedding model** — Same model for index and query? Same `dimensions`?
3. **Similarity** — Is the right chunk in top-k at all? (retrieval eval)
4. **Keywords** — Does the query need exact token match (error codes)?
5. **Generation** — If retrieval is correct, fix prompts — not embeddings.

```
Bad answer → Was right text retrieved? ─No→ fix chunks / embed / index
                              └─Yes→ fix prompt / LLM
```

---

## 10. Demonstration Setup

Replace the placeholder API key before running. Cells below reuse `embed_texts()` and `cosine_similarity()`.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"


def embed_texts(texts: list[str]) -> list[list[float]]:
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]


def cosine_similarity(a: list[float], b: list[float]) -> float:
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))


print("Ready.")

---

## 11. Demonstration: Anatomy of an Embedding Vector

Inspect dimensionality, value range, and API usage metadata.

In [ ]:
sample = "Password reset instructions for enterprise SSO users."
response = client.embeddings.create(model=EMBED_MODEL, input=[sample])
vector = response.data[0].embedding

arr = np.array(vector)
print(f"Model:       {EMBED_MODEL}")
print(f"Text:        {sample}")
print(f"Dimensions:  {len(vector)}")
print(f"Tokens used: {response.usage.total_tokens}")
print(f"Min / max:   {arr.min():.4f} / {arr.max():.4f}")
print(f"Mean / std:  {arr.mean():.4f} / {arr.std():.4f}")
print(f"First 8:     {arr[:8]}")

Values are small floats — not probabilities. The vector is a compressed fingerprint of meaning for similarity comparisons.

---

## 12. Demonstration: Paraphrase vs Related vs Unrelated

Five sentences spanning paraphrase, same-topic different wording, and unrelated content.

In [ ]:
anchor = "How do I reset my password?"
candidates = [
    ("paraphrase", "I forgot my login credentials and need access again."),
    ("related", "Two-factor authentication is required after password change."),
    ("unrelated", "The weather in Paris is sunny today."),
    ("unrelated", "Alembic database migration revision files."),
]

all_texts = [anchor] + [c[1] for c in candidates]
vectors = embed_texts(all_texts)
anchor_vec = vectors[0]

print(f"Anchor: {anchor}\n")
for (label, text), vec in zip(candidates, vectors[1:]):
    sim = cosine_similarity(anchor_vec, vec)
    print(f"  [{label:10s}] {sim:.4f}  {text}")

Expect **paraphrase** highest, **related** moderate, **unrelated** lowest. "Related" (2FA) shares auth domain but is not a paraphrase of password reset.

---

## 13. Demonstration: Pairwise Similarity Matrix

View all pairwise cosines for a small set — useful when tuning thresholds.

In [ ]:
phrases = [
    "reset password",
    "forgot login",
    "JWT authentication",
    "schema migration",
]

vecs = np.array(embed_texts(phrases))
norms = np.linalg.norm(vecs, axis=1, keepdims=True)
unit = vecs / norms
sim_matrix = unit @ unit.T

print("Cosine similarity matrix:")
print(f"{'':18s}" + "".join(f"{p[:12]:>14s}" for p in phrases))
for i, row_label in enumerate(phrases):
    row = "".join(f"{sim_matrix[i, j]:14.3f}" for j in range(len(phrases)))
    print(f"{row_label[:18]:18s}{row}")

---

## 14. Demonstration: Short vs Long Input

Same topic — terse query vs detailed paragraph. Vectors differ but should stay relatively close.

In [ ]:
short = "database migrations"
long = (
    "Alembic is a lightweight database migration tool for SQLAlchemy. "
    "Developers generate revision scripts and apply them with upgrade head "
    "in local, staging, and production environments."
)

v_short, v_long = embed_texts([short, long])
print(f"cosine(short, long): {cosine_similarity(v_short, v_long):.4f}")
print("\nChunking lesson: store the LONG form in the index; users often query with SHORT phrases.")

---

## 15. Demonstration: Batch Embedding

Multiple strings in one API call — fewer round trips during ingest.

In [ ]:
batch = [
    "FastAPI dependency injection.",
    "SQLAlchemy session management.",
    "Pytest fixtures for integration tests.",
]

response = client.embeddings.create(model=EMBED_MODEL, input=batch)
print(f"Inputs: {len(batch)}  |  Tokens: {response.usage.total_tokens}")
for item in response.data:
    print(f"  index={item.index}  dim={len(item.embedding)}  preview={batch[item.index][:40]}...")

---

## 16. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| One embedding per huge document | Vague retrieval | Chunk first |
| Different models for index vs query | Broken similarity | Same model + dimensions |
| Ignoring keyword search | Miss error codes / SKUs | Hybrid search |
| No retrieval eval | Tune blindly | Labeled query set (nb 09) |
| Treating high similarity as truth | Wrong but plausible chunks | Ground LLM + cite sources |

---

## 17. Summary

Embeddings map text to dense vectors $\mathbf{e} \in \mathbb{R}^d$ so that **semantic similarity ≈ geometric closeness**. They solve the paraphrase and synonym failures of sparse keyword methods.

From Word2Vec to API models, the shift is toward **contextual, span-level** vectors you obtain via HTTP — no training in app code. In production, embeddings sit in a pipeline: **chunk → embed → store → retrieve** before any LLM call.

Demonstrations showed vector anatomy, paraphrase vs unrelated similarity, a pairwise matrix, short-vs-long inputs, and batch API usage.

Next: **02. Similarity Search and Distance Metrics** — cosine, dot product, L2, and nearest-neighbor search in detail.